#### Embedding Models

- An embedding converts text into a vector (a list of numbers) that captures semantic meaning. Similar meaning → similar vectors → close together in vector space.

##### First understand the problem 
- suppose we have these sentences 
```
1. I love dogs.
2. I like puppies.
3. Python is a programming language.
```
- To Humans:
    - Sentance 1 and 2 are similar.
    - Sentence 3 is completely different.

- but here computer see
```
"I love dogs"
"I like puppies"
"Python is a programming language"
```
as a plain text , they dont understand meaning of the words.

#### So here How do computer understand Meaning?
- we convert text into numbers this process is called embedding.
```
Text
 ↓
Embedding Model
 ↓
Vector (Numbers)
```
- these numbers are called Embeddings.

#### What is an Embedding?
- An embedding is a numerical vecotr representation of text that captures its semantic meaning.

- example ```i love dogs``` might become ```[0.12,-0.34,0.89,0.55,...]  (Usually 768, 1024, or 1536 dimensions(size of embedding))
- another exampe: ```i like puppies``` it becomes ```[0.10, -0.30, 0.85, 0.58, ...]``` if you observe these vectors are closed together.

###### Why?
- because both sentences mean alsmost the same thing. 
- here meanwhile ```python is programming langauge``` it may become ```[-0.90, 0.50, -0.20, 0.70, ...]``` it lies far away.

##### Visual Representation.
```
Dog
 Puppy

        (close)

-------------------------

Python Programming

        (far)
```
- Emebedding models place semantically similar texts close together.

#### here why do we need Embeddings?
- here suppose user asks: ```tell me about caninies```
- document contains: ```"Dogs are loyal animals."```
- if we do keyword search: ```canines ≠ dogs```  it may fail the users question. 
- in the above case embedding search will understands ```canines~~dogs``` thus retrival get sucess and user will got the response back. this is the reason behind the embedding models exists. 

#### where embeding fit in RAG
```
PDF
 ↓
Document Loader
 ↓
Text Splitter
 ↓
Chunks
 ↓
Embedding Model
 ↓
Vectors
 ↓
Vector Database
 ↓
Retriever
 ↓
LLM
```


##### How Embedding Model works.
- chunk : ```Apache Spark is a distributed processing framework.```
- Embedding : the chunk will sent to the embedding model
- Vecotr : Model will convert to vecotrs ```[0.23, -0.55, 0.88, 0.67, ...]``` and then stored in the vecotr database.
- Retrival: when user asks ```what is spark``` then question is also converted into vecotr then similarity search finds nearest chunk and it will give the response 


#### NOTE 
- embedding models do not generate text.
- chat models: ```input --> output text```
- embedding models: ```input text ---> output numbers```



##### chat models vs embedding models 
| Chat Model          | Embedding Model       |
| ------------------- | --------------------- |
| Generates text      | Generates vectors     |
| GPT, Claude, Gemini | OpenAIEmbeddings, BGE |
| Used for answering  | Used for retrieval    |
| Output = words      | Output = numbers      |



#### Dimensions 
- Every embedding model produces vectors of fixed sizes.
- An embedding dimension represents the number of numeric values(floating-point-numbers) in single vecotr array generated by an embedding model. 
- if a model outputs a vecotr with 1024 numbers, it has a dimensionality of 1024, which can be thought of as a coordinate point in a 1024- dimensional space where each dimension tracks a unique semantic comcept or nuance of the language

#### Why Dimension Size Matters
- *Higher Dimensions* (e.g., 1536, 3072): Capture deep semantic complexity, subtle nuances, and multi-lingual details. However, they require more memory, increase vector database costs, and slow down mathematical similarity calculations.
- *Lower Dimensions* (e.g., 384, 768): Offer lightning-fast search speeds and minimal memory footprints. However, they may struggle to differentiate between highly specific or technical contexts.


###### Examples 
| Model                         | Dimensions |
| ----------------------------- | ---------- |
| OpenAI text-embedding-3-small | 1536       |
| MiniLM                        | 384        |
| BGE Base                      | 768        |
| BGE Large                     | 1024       |


#### Embedding model integrations
- Embedding models transform raw text—such as a sentence, paragraph, or tweet—into a fixed-length vector of numbers that captures its semantic meaning. These vectors allow machines to compare and search text based on meaning rather than exact words.
- In practice, this means that texts with similar ideas are placed close together in the vector space. For example, instead of matching only the phrase “machine learning”, embeddings can surface documents that discuss related concepts even when different wording is used.
- How it works:
    1. *Vectorization* — The model encodes each input string as a high-dimensional vector.
    2. *Similarity scoring* — Vectors are compared using mathematical metrics to measure how closely related the underlying texts are.

- Similarity Metrics
- several metrics are commonly used to compare embeddings:
    * *Cosine similarity* — measures the angle between two vectors.
    * *Euclidean distance* — measures the straight-line distance between points.
    * *Dot product* — measures how much one vector projects onto another.



In [ ]:
# Here’s an example of computing cosine similarity between two vectors:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

similarity = cosine_similarity(query_embedding, document_embedding)
print("Cosine Similarity:", similarity)

#### Interface
- LangChain provides a standard interface for text embedding models (e.g., OpenAI, Cohere, Hugging Face) via the Embeddings interface.
- This is an interface meant for implementing text embedding models.
- Text embedding models are used to map text to a vector (a point in n-dimensional space).
- Texts that are similar will usually be mapped to points that are close to each other in this space. The exact details of what's considered "similar" and how "distance" is measured in this space are dependent on the specific embedding model
- This abstraction contains a method for embedding a list of documents and a method for embedding a query text. The embedding of a query text is expected to be a single vector, while the embedding of a list of documents is expected to be a list of vectors.


- Two main methods are available:
    - embed_documents(texts: List[str]) → List[List[float]]: Embeds a list of documents.
    - embed_query(text: str) → List[float]: Embeds a single query.

#### Common Embedding Models 

1. 1 — OpenAI Embeddings
- The most common starting point. text-embedding-3-small is fast and cheap; text-embedding-3-large gives higher quality at higher cost.


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",   # or text-embedding-3-large
    openai_api_key="sk-..."           # or set OPENAI_API_KEY env var
)

# Embed a single query
query_vec = embeddings.embed_query("What is LangChain?")
print(f"Dimensions: {len(query_vec)}")   # 1536 for small, 3072 for large
print(query_vec[:5])                     # [-0.012, 0.043, -0.071, ...]

# Embed multiple documents at once (batched internally)
texts = [
    "LangChain is a framework for LLM apps.",
    "It supports RAG, agents, and memory.",
    "The Eiffel Tower is in Paris, France."
]
doc_vecs = embeddings.embed_documents(texts)
print(f"Got {len(doc_vecs)} vectors, each of size {len(doc_vecs[0])}")

In [ ]:
# Dimension reduction (cost saving)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=256    # reduce from 3072 → 256 with minimal quality loss
)

2. HuggingFace Embeddings (free, local)
- No API key. Models run locally on your machine. Best for offline use, cost-sensitive projects, or when data cannot leave your infrastructure.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model runs locally — first call downloads ~90MB
    model_kwargs={"device": "cpu"},     # or "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

vec = embeddings.embed_query("What is LangChain?")
print(f"Dimensions: {len(vec)}")   # 384 for MiniLM

doc_vecs = embeddings.embed_documents([
    "LangChain is a framework for LLM apps.",
    "The Eiffel Tower is in Paris."
])
print(doc_vecs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2726.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensions: 384
[[-0.019657906144857407, -0.053824957460165024, -0.0031295798253268003, -0.11404190212488174, 0.019604837521910667, 0.005661375354975462, -0.037501994520425797, 0.06319907307624817, 0.02877919375896454, -0.02504006400704384, 0.004208484664559364, 0.017287826165556908, 0.004972127731889486, 0.006916115991771221, 0.07196329534053802, 0.03484691306948662, 0.019509827718138695, 0.08814455568790436, 0.06537634879350662, -0.06482397764921188, -0.009389588609337807, -0.0007707225740887225, 0.03895086422562599, 0.050539035350084305, 0.06432543694972992, -0.07439292222261429, 0.011761719360947609, 0.031798504292964935, 0.06646306067705154, -0.0011766351526603103, 0.03853398934006691, 0.11708736419677734, -0.03405130282044411, 0.005492021795362234, -0.10947874933481216, 0.06185751035809517, -0.01589660905301571, -0.0538400299847126, -0.08317093551158905, -0.07709842175245285, -0.059190016239881516, 0.01457050908356905, 0.0163918174803257, -0.021070344373583794, 0.0353730507194995

##### 4.Caching Embeddings (production must-have)
- Embedding the same document twice wastes money and time. Use CacheBackedEmbeddings to store results on disk.

In [11]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_huggingface import HuggingFaceEmbeddings

# Underlying embedder
base_embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Persistent cache on disk
store = LocalFileStore("./embedding_cache/")

# Wrap with cache
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=base_embedder,
    document_embedding_cache=store,
    namespace=base_embedder.model_name    # keeps caches separated per model
)

texts = ["LangChain is a framework.", "RAG retrieves documents."]

# First call — hits the API
vecs1 = cached_embeddings.embed_documents(texts)
print('vector reperesentation at the start',vecs1)

# Second call — served from disk cache, zero API cost
vecs2 = cached_embeddings.embed_documents(texts)
print(vecs2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2933.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vector reperesentation at the start [[-0.014280404895544052, -0.03338014706969261, -0.020167304202914238, -0.07670090347528458, -0.01619408279657364, 0.027854623273015022, -0.002733987057581544, 0.0592355839908123, 0.03875672072172165, -0.0211221594363451, 0.048431333154439926, 0.03896987438201904, -0.03734789788722992, 0.04626191034913063, 0.02719435840845108, -0.03352391719818115, -0.06217075511813164, 0.07084985822439194, 0.03291005641222, -0.08015158772468567, -0.06429590284824371, -0.005027627106755972, 0.025444958359003067, 0.0502462312579155, 0.02817692421376705, -0.07070650905370712, -0.009372537024319172, -0.04018663987517357, 0.06437557935714722, 0.001468344242312014, 0.02802024595439434, 0.11118537187576294, -0.021545294672250748, -0.0087139206007123, -0.10295466333627701, 0.1265498399734497, 0.01596686989068985, -0.05386810377240181, -0.030515385791659355, -0.04695698991417885, -0.08938727527856827, 0.009748784825205803, 0.016290904954075813, -0.03779877722263336, 0.0407664

c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


 ### 5.Measuring Similarity (cosine similarity)
- After embedding, similarity is measured by the angle between vectors. Cosine similarity of 1.0 = identical meaning, 0.0 = unrelated, -1.0 = opposite.


In [12]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sentences = [
    "I love dogs",
    "I adore puppies",
    "The stock market crashed today",
    "My pet labrador is adorable"
]

vecs = embeddings.embed_documents(sentences)

# Compare each pair
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(vecs[i], vecs[j])
        print(f"{sentences[i]!r} <-> {sentences[j]!r}")
        print(f"  similarity: {sim:.3f}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2288.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'I love dogs' <-> 'I adore puppies'
  similarity: 0.683

'I love dogs' <-> 'The stock market crashed today'
  similarity: 0.042

'I love dogs' <-> 'My pet labrador is adorable'
  similarity: 0.589

'I adore puppies' <-> 'The stock market crashed today'
  similarity: 0.007

'I adore puppies' <-> 'My pet labrador is adorable'
  similarity: 0.607

'The stock market crashed today' <-> 'My pet labrador is adorable'
  similarity: 0.033



In [ ]:
#### 9.Choosing the right embedding model.
# Development / experimenting — fast and free
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Production, English only — best balance of cost/quality
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Production, multilingual (Kannada, Hindi, English mix)
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

# Production, highest quality needed
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


#### Note :
- embed_documents() is for ingestion, embed_query() is for retrieval — never swap them. Always wrap your embedder with CacheBackedEmbeddings in production to avoid re-embedding the same content on every restart. The embedding model you choose at ingestion time must be the same one you use at query time — switching models invalidates your entire vector store. And dimensionality is not everything — text-embedding-3-small at 1536 dims often outperforms a poorly-tuned 3072-dim model on domain-specific data.

### Overview.
Embedding models convert text into dense numerical vectors that capture semantic meaning. These vectors enable similarity search and retrieval in RAG systems. LangChain supports multiple embedding providers such as OpenAI, HuggingFace, BGE, Cohere, and Gemini, and commonly uses embed_documents() for document chunks and embed_query() for user queries before performing similarity search in vector databases